In [4]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
import os


# -------------------------
# TODO 1: Load Documents
# -------------------------
def load_documents():
    loader = TextLoader("data/notes.txt")
    return loader.load()


# -------------------------
# TODO 2: Custom Chunking
# -------------------------
def custom_chunking(docs):

    chunks = []

    for doc in docs:

        # Split by section separator
        sections = doc.page_content.split("\n\n---\n\n")

        for section in sections:

            section = section.strip()

            if not section:
                continue

            # Extract section title (first line)
            lines = section.split("\n")
            title = lines[0]

            # Remaining content
            content = "\n".join(lines[1:])

            chunks.append(
                Document(
                    page_content=content,
                    metadata={
                        "section": title,
                        "source": "notes.txt"
                    }
                )
            )

    return chunks


# -------------------------
# TODO 3: Create Vector Store
# -------------------------
def create_vector_store(chunks):

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small", 
                                  openai_api_key=os.getenv("OPENAI_API_KEY"))

    return Chroma.from_documents(
        chunks,
        embedding=embeddings,
        collection_name="rag_chunking_demo"
    )


# -------------------------
# TODO 4: Retrieval
# -------------------------
def retrieve_context(vector_store, query):

    # Retrieve top 3 relevant chunks
    return vector_store.similarity_search(query, k=3)


# -------------------------
# TODO 5: Context Builder
# -------------------------
def build_context(docs):

    context_parts = []

    for doc in docs:
        section = doc.metadata.get("section", "Unknown")
        context_parts.append(f"[{section}]\n{doc.page_content}")

    return "\n\n".join(context_parts)


# -------------------------
# TODO 6: LLM Response
# -------------------------
def generate_answer(context, query):

    llm = ChatOpenAI(model="gpt-4o", temperature=0, openai_api_key=os.getenv("OPENAI_API_KEY"))

    messages = [
        SystemMessage(content="You are a senior product strategist."),
        HumanMessage(content=f"""
Use the context below to answer the question.

Context:
{context}

Question:
{query}
""")
    ]

    return llm.invoke(messages).content


# -------------------------
# TODO 7: Full Pipeline
# -------------------------
def rag_pipeline(query):

    docs = load_documents()

    chunks = custom_chunking(docs)

    vector_store = create_vector_store(chunks)

    retrieved_docs = retrieve_context(vector_store, query)

    context = build_context(retrieved_docs)

    answer = generate_answer(context, query)

    return answer


# -------------------------
# Run Test
# -------------------------
query = "What pricing strategy works best for enterprise customers?"

response = rag_pipeline(query)

print(response)

For enterprise customers, the best pricing strategy involves tiered pricing with a focus on reliability, security, and support. This approach allows for segmentation based on usage, features, and support levels, which aligns with the priorities of enterprise customers who value guaranteed uptime and dedicated account management over cost. Additionally, offering trials instead of freemium models can improve conversion rates, as enterprise customers require security, compliance, and onboarding support that free tiers typically do not provide.
